# Fine-tuned DistilBERT spam classifier

**Project:** Secure Local Mail System with ML-Based Spam Detection Under Adversarial Conditions  
**Author:** Abdulla AlBassam  
**Module:** KV6013, Individual Computing Project  
**Supervisor:** Longzhi Yang  

Fine-tunes DistilBERT on the Enron spam corpus for binary spam/ham classification, then writes the trained model and a metrics CSV out for comparison against the baseline. The point of running this on Colab is the free T4 GPU; on the dev laptop training would be measured in hours rather than minutes.

For the comparison against the baseline (notebook 02) to be honest, every variable that isn't "the model" must match. So:
- Same dataset (Enron, 33,662 emails after cleaning).
- Same train/test split (80/20 stratified, `random_state=42`).
- Same text preprocessing (`clean_text()` from `preprocessing.py`, copied verbatim into this notebook because Colab can't see the project tree).
- Same evaluation metrics, computed the same way.

**Environment:** Google Colab with GPU runtime (T4 recommended)

## 1. Install dependencies and check GPU

`transformers` carries the pre-trained DistilBERT weights and tokeniser. `accelerate` handles mixed-precision (FP16) training so the T4 doesn't run out of memory and so each step is roughly twice as fast. Colab's free T4 has 15 GB of VRAM, comfortably enough for a 67M-parameter model at batch size 16.

**Before running, make sure GPU is enabled:**
1. Click **Runtime** then **Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU**
3. Click **Save**

If you forget this step, the notebook will still run, but on CPU it would take hours instead of ~20 minutes.

In [ ]:
# These are not in the default Colab image, so install them every run
!pip install transformers datasets accelerate -q

import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    # Better to bail loudly than to silently train on CPU and discover an
    # hour later that nothing has progressed.
    print('WARNING: No GPU detected. Training will be very slow.')
    print('Go to Runtime > Change runtime type > T4 GPU')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')

## 2. Upload dataset

Upload `enron_spam_data.csv` (49 MB), the same file that lives at `ml/data/raw/enron_spam_data.csv` in the project tree. Colab can't see the project files, so each run starts with a fresh upload.

In [ ]:
import os
from google.colab import files

if not os.path.exists('enron_spam_data.csv'):
    print('Please upload enron_spam_data.csv (49MB)')
    print('This is located at: ml/data/raw/enron_spam_data.csv in your project folder')
    uploaded = files.upload()
    print(f'\nUploaded: {list(uploaded.keys())}')
else:
    print('Dataset already uploaded.')

print(f'File size: {os.path.getsize("enron_spam_data.csv") / 1e6:.1f} MB')

## 3. Preprocessing functions

Pasted verbatim from `ml/preprocessing.py`. Colab does not have access to the project tree, so the alternative would be uploading the file separately, which adds a step that's easy to forget. Inlining is uglier but robust.

Identical preprocessing matters. The whole comparison against the baseline rests on both models seeing the same cleaned text; the only difference between the two pipelines should be how that text is turned into features (TF-IDF vocabulary on one side, WordPiece tokenisation plus contextual embeddings on the other). If this code drifts from `ml/preprocessing.py`, the comparison is no longer apples-to-apples.

In [ ]:
import re
import csv


def load_enron_data(filepath):
    """
    Load the Enron spam dataset from CSV.

    Handles the multi-line message bodies in the dataset by increasing
    the CSV field size limit. Some emails contain very large bodies
    (up to ~228K characters) that exceed Python's default limit.
    """
    csv.field_size_limit(10 ** 7)
    df = pd.read_csv(filepath, encoding='utf-8', on_bad_lines='skip')
    return df


def clean_text(text):
    """
    Clean a single email text string for ML processing.

    Steps: lowercase -> remove headers -> remove emails -> remove URLs
    -> remove HTML -> remove non-alpha -> collapse whitespace.
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""

    text = text.lower()

    # Remove common email header lines
    text = re.sub(
        r'^(from|to|cc|bcc|subject|date|sent|received|reply-to|'
        r'content-type|mime-version|x-mailer|x-originating-ip):.*$',
        '', text, flags=re.MULTILINE
    )

    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # Remove non-alphabetic characters (keep spaces)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def combine_subject_message(row):
    """
    Combine Subject and Message fields into a single text string.
    Subject is prepended so its terms appear at the start.
    """
    subject = clean_text(row.get('Subject', ''))
    message = clean_text(row.get('Message', ''))
    return f"{subject} {message}".strip()


def preprocess_dataset(df):
    """
    Full preprocessing pipeline for the Enron spam dataset.
    Combines subject + message, encodes labels (spam=1, ham=0),
    and drops empty rows.
    """
    df = df.copy()
    df['text'] = df.apply(combine_subject_message, axis=1)
    df['label'] = (df['Spam/Ham'].str.strip().str.lower() == 'spam').astype(int)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    return df[['text', 'label']]


print('Preprocessing functions loaded (identical to ml/preprocessing.py)')

## 4. Load and preprocess data

Run the pipeline and assert that the row count matches the baseline before going any further. Drift here means I'm training on a different dataset than the baseline saw, and any later comparison would be meaningless. Expected numbers (taken from notebook 02):
- Raw: 33,716 emails
- After cleaning: 33,662 (54 rows dropped because cleaned text was empty)
- Balance: ~50.9% spam / ~49.1% ham

In [ ]:
# Load and preprocess
df_raw = load_enron_data('enron_spam_data.csv')
print(f'Raw dataset: {len(df_raw):,} rows')
print(f'Columns: {list(df_raw.columns)}')

df = preprocess_dataset(df_raw)
print(f'\nAfter preprocessing: {len(df):,} rows')
print(f'\nLabel distribution:')
print(f'  Ham (0):  {(df["label"] == 0).sum():,} ({(df["label"] == 0).mean() * 100:.1f}%)')
print(f'  Spam (1): {(df["label"] == 1).sum():,} ({(df["label"] == 1).mean() * 100:.1f}%)')

# Hard assert: this number must match the baseline notebook exactly.
# If it doesn't, stop and figure out why before training.
assert len(df) == 33662, f'Expected 33,662 rows but got {len(df):,}'
print('\nRow count matches baseline (33,662).')

## 5. Train/test split

Same 80/20 stratified split as the baseline notebook, with the same `random_state=42`. That guarantees both models train on the same 26,929 samples and are evaluated on the same 6,733 samples. If `random_state` differed, the two models would be looking at different test sets and the comparison would be meaningless.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f'Training set: {len(X_train):,} samples')
print(f'Test set:     {len(X_test):,} samples')
print(f'\nTraining label distribution:')
print(f'  Ham:  {(y_train == 0).sum():,} ({(y_train == 0).mean() * 100:.1f}%)')
print(f'  Spam: {(y_train == 1).sum():,} ({(y_train == 1).mean() * 100:.1f}%)')

# Hard asserts again. The baseline notebook produces exactly these
# numbers, and the comparison is only valid if this notebook matches.
assert len(X_train) == 26929, f'Expected 26,929 train but got {len(X_train):,}'
assert len(X_test) == 6733, f'Expected 6,733 test but got {len(X_test):,}'
print('\nTrain/test split matches baseline (26,929 / 6,733).')

## 6. Tokenisation

DistilBERT uses WordPiece tokenisation, which splits any word into subword units that are in its vocabulary. Unlike TF-IDF (which has a fixed whole-word vocabulary), WordPiece will never see a true out-of-vocabulary token: it just decomposes unknown words into known pieces. For example, "unsubscribe" tokenises as `["un", "##sub", "##scribe"]`. This is one of the reasons transformer models are expected to be more robust to character-level adversarial attacks: a typo'd spam keyword still tokenises into something sensible.

I'm using `distilbert-base-uncased`. The "uncased" half matters: it lowercases everything internally, which lines up with the lowercasing my `clean_text()` already does. Using the cased version would mean the model sees a different cased distribution than the cleaned input would naturally produce.

`max_length=256` is the truncation point. The cell below prints what fraction of emails get truncated at that length so I can be honest about it in the report. Putting Subject before Message in `combine_subject_message` is what protects the most signal-rich part from truncation: even on a 50K-character spam email, the subject line and the first chunk of the body always survive.

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

MAX_LENGTH = 256

# Sample 1,000 training texts to estimate the truncation rate. Reading
# this number is what tells me whether 256 is reasonable for this corpus.
print('Analysing token lengths on first 1,000 training texts...')
sample_lengths = [len(tokenizer.encode(text, truncation=False)) for text in X_train[:1000]]

print(f'\nToken length statistics (sample of 1,000):')
print(f'  Mean:   {np.mean(sample_lengths):.0f} tokens')
print(f'  Median: {np.median(sample_lengths):.0f} tokens')
print(f'  95th percentile: {np.percentile(sample_lengths, 95):.0f} tokens')
print(f'  Max:    {np.max(sample_lengths)} tokens')
truncated = sum(1 for l in sample_lengths if l > MAX_LENGTH)
print(f'  Truncated at {MAX_LENGTH}: {truncated} / 1,000 ({truncated / 10:.1f}%)')

print(f'\nTokenising training set ({len(X_train):,} texts)...')
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors='pt'
)
print(f'Training input shape: {train_encodings["input_ids"].shape}')

print(f'\nTokenising test set ({len(X_test):,} texts)...')
test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH,
    return_tensors='pt'
)
print(f'Test input shape: {test_encodings["input_ids"].shape}')
print('\nTokenisation complete.')

## 7. PyTorch dataset wrapper

HuggingFace's `Trainer` wants a torch `Dataset` rather than raw tensors, because it pulls items lazily and shards across workers. The wrapper below is the smallest thing that works: pull tokenised tensors out of `encodings` for each index, attach the label, return a dict that `Trainer` knows how to feed to the model.

In [ ]:
class SpamDataset(torch.utils.data.Dataset):
    """
    Minimal torch Dataset over already-tokenised inputs.

    __getitem__ returns a dict with input_ids (token indices),
    attention_mask (1 for real tokens, 0 for padding), and labels
    (0 = ham, 1 = spam). HuggingFace Trainer feeds these straight to
    DistilBertForSequenceClassification.
    """
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = SpamDataset(train_encodings, y_train)
test_dataset = SpamDataset(test_encodings, y_test)

print(f'Training dataset: {len(train_dataset):,} samples')
print(f'Test dataset:     {len(test_dataset):,} samples')

# Quick sanity check on a single sample
sample = train_dataset[0]
print(f'\nSample item keys: {list(sample.keys())}')
print(f'Input IDs shape: {sample["input_ids"].shape}')
print(f'Attention mask shape: {sample["attention_mask"].shape}')
print(f'Label: {sample["labels"].item()}')

## 8. Model and training configuration

Load `distilbert-base-uncased` and slap a 2-class classification head on top. The whole stack (about 67 million parameters) is fine-tuned, not frozen; for a corpus this size and a task this specific, full fine-tuning produces noticeably better numbers than frozen-encoder + linear probing.

### Hyperparameter choices

| Parameter | Value | Why this value |
|-----------|-------|--------|
| Learning rate | 2e-5 | The original BERT paper recommends 2e-5 to 5e-5. Sticking to the low end is safer for training stability on a small fine-tune. |
| Batch size | 16 | Fits in 15 GB of T4 VRAM with FP16 and headroom to spare. The standard DistilBERT recipe. |
| Epochs | 3 | Standard for BERT/DistilBERT fine-tuning. Going further usually starts overfitting on tasks this size. |
| Warmup | 10% of total steps | Prevents the optimiser from making huge updates at step 0 when the classification head is still random. |
| Weight decay | 0.01 | AdamW regularisation, light hand on the brakes. |
| FP16 | True | Mixed precision halves memory and roughly halves step time on a T4. Free win on this hardware. |
| Seed | 42 | Reproducibility. Note: CUDA ops are not fully deterministic, so reruns will be very close but not bit-identical. |

In [ ]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load pre-trained DistilBERT with a 2-class classification head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model.to(device)

# Calculate training steps for warmup schedule
BATCH_SIZE = 16
EPOCHS = 3
total_steps = (len(train_dataset) // BATCH_SIZE) * EPOCHS
warmup_steps = int(total_steps * 0.1)

print(f'Model: distilbert-base-uncased')
print(f'Total parameters:     {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'\nTraining configuration:')
print(f'  Epochs:        {EPOCHS}')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  Learning rate: 2e-5')
print(f'  Total steps:   {total_steps:,}')
print(f'  Warmup steps:  {warmup_steps}')
print(f'  Weight decay:  0.01')
print(f'  FP16:          {torch.cuda.is_available()}')


def compute_metrics(eval_pred):
    """Compute metrics during training for epoch-level evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir='./distilbert_checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    seed=42,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print('\nModel and trainer configured.')

## 9. Train the model

3 epochs of fine-tuning. On a T4 this takes roughly 15 to 25 minutes; if the cell finishes in under 5 minutes, GPU isn't actually engaged.

What the trainer logs:
- Training loss every 100 steps.
- Eval metrics (accuracy, precision, recall, F1) at the end of each epoch.
- Best checkpoint is auto-selected by F1, since this corpus is balanced enough that F1 is the right tiebreaker.

In [ ]:
print('Starting training...\n')
train_result = trainer.train()

print(f'\n{"=" * 50}')
print(f'Training finished.')
print(f'Training loss:    {train_result.training_loss:.4f}')
print(f'Training time:    {train_result.metrics["train_runtime"]:.0f} seconds')
print(f'Samples/second:   {train_result.metrics["train_samples_per_second"]:.1f}')
print(f'{"=" * 50}')

## 10. Test set evaluation

Final evaluation on the held-out 6,733 test emails. Nothing about training has been allowed to peek at this set, so these numbers are honest. They're the figures that go into the dissertation alongside the baseline; same samples, same metrics, same preprocessing, only the model differs.

**Baseline numbers for reference (from notebook 02):**
- Accuracy: 99.09% | Precision: 98.53% | Recall: 99.71% | F1: 99.11% | ROC-AUC: 99.91%

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)
from scipy.special import softmax

# Get predictions on the test set
predictions = trainer.predict(test_dataset)

# Convert logits to class predictions
y_pred = np.argmax(predictions.predictions, axis=-1)

# Get probabilities for ROC-AUC (apply softmax to logits)
y_pred_proba = softmax(predictions.predictions, axis=1)[:, 1]

# Calculate all metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

print(f'\n{"=" * 40}')
print(f'  DistilBERT Results')
print(f'{"=" * 40}')
print(f'  Accuracy:  {acc:.4f} ({acc * 100:.2f}%)')
print(f'  Precision: {prec:.4f} ({prec * 100:.2f}%)')
print(f'  Recall:    {rec:.4f} ({rec * 100:.2f}%)')
print(f'  F1 Score:  {f1:.4f} ({f1 * 100:.2f}%)')
print(f'  ROC-AUC:   {auc:.4f} ({auc * 100:.2f}%)')
print(f'{"=" * 40}')

print(f'\nBaseline comparison:')
print(f'{"Metric":<12} {"Baseline":<12} {"DistilBERT":<12} {"Difference":<12}')
print(f'{"-" * 48}')
baseline = {'Accuracy': 0.9909, 'Precision': 0.9853, 'Recall': 0.9971, 'F1': 0.9911, 'ROC-AUC': 0.9991}
distilbert = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': auc}
for metric in baseline:
    bl = baseline[metric]
    db = distilbert[metric]
    diff = db - bl
    sign = '+' if diff >= 0 else ''
    print(f'{metric:<12} {bl:<12.4f} {db:<12.4f} {sign}{diff:.4f}')

## 11. Confusion matrix

Plotted in the same style as the baseline confusion matrix so the two can sit side by side in the report without one looking like an outlier.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Ham', 'Spam'],
    yticklabels=['Ham', 'Spam'],
    ax=ax, linewidths=0.5, linecolor='gray'
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('DistilBERT Model: Confusion Matrix', fontsize=14)

plt.tight_layout()
plt.savefig('distilbert_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Ham correctly classified):   {tn:,}')
print(f'False Positives (Ham misclassified as Spam):  {fp:,}')
print(f'False Negatives (Spam misclassified as Ham):  {fn:,}')
print(f'True Positives  (Spam correctly classified):  {tp:,}')

print(f'\nBaseline confusion matrix for comparison:')
print(f'  TN: 3,258 | FP: 51 | FN: 10 | TP: 3,414')

## 12. ROC curve

The ROC curve shows the trade-off between true-positive rate and false-positive rate as the decision threshold sweeps across [0, 1]. AUC summarises overall discrimination: 1.0 is perfect, 0.5 is no better than random. For spam classification, a high AUC matters because operators tune the threshold based on tolerance for false positives, and a higher-AUC model gives them more room to manoeuvre.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'DistilBERT (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random (AUC = 0.5)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('DistilBERT Model: ROC Curve', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('distilbert_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'ROC-AUC: {auc:.4f}')

## 13. Training loss curve

The training loss over steps. Logistic Regression converges in essentially one step on this size of dataset; transformers train iteratively via SGD, so the loss curve is something to actually inspect.

What I'm looking for: a smoothly decreasing curve. A plateau early in training would suggest the learning rate is too low; a curve that climbs after dropping would suggest overfitting or that the LR is too high. Healthy fine-tuning runs of DistilBERT on a corpus this size usually drop sharply in the first half of epoch 1 and then settle into a slow decline.

In [ ]:
# Pull (step, loss) pairs out of Trainer's log history. Eval entries
# also live in log_history but lack the 'loss' key, so the filter below
# leaves them out automatically.
log_history = trainer.state.log_history
train_losses = [(entry['step'], entry['loss']) for entry in log_history if 'loss' in entry]

if train_losses:
    steps, losses = zip(*train_losses)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(steps, losses, color='darkorange', lw=2, alpha=0.8)
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title('DistilBERT: Training Loss Curve', fontsize=14)
    ax.grid(True, alpha=0.3)

    # Add epoch markers
    steps_per_epoch = len(train_dataset) // BATCH_SIZE
    for epoch in range(1, EPOCHS + 1):
        epoch_step = epoch * steps_per_epoch
        ax.axvline(x=epoch_step, color='gray', linestyle=':', alpha=0.5)
        ax.text(epoch_step, max(losses) * 0.95, f'Epoch {epoch}',
                ha='center', fontsize=9, color='gray')

    plt.tight_layout()
    plt.savefig('distilbert_training_loss.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f'Initial loss: {losses[0]:.4f}')
    print(f'Final loss:   {losses[-1]:.4f}')
else:
    print('No training loss data available.')

## 14. Export results CSV

Save metrics in the same column order as `evaluation/baseline_results.csv` so the comparison logic downstream can just concatenate the two files instead of doing schema gymnastics.

In [ ]:
results = pd.DataFrame([{
    'Model': 'Fine-Tuned DistilBERT',
    'Accuracy': round(acc, 4),
    'Precision': round(prec, 4),
    'Recall': round(rec, 4),
    'F1': round(f1, 4),
    'ROC-AUC': round(auc, 4),
    'Train Size': len(X_train),
    'Test Size': len(X_test),
    'Features': 'contextual embeddings (768-dim)',
    'True Negatives': int(tn),
    'False Positives': int(fp),
    'False Negatives': int(fn),
    'True Positives': int(tp)
}])

results.to_csv('distilbert_results.csv', index=False)
print('Results saved to distilbert_results.csv')
print()
print(results.T.to_string(header=False))

## 15. Save trained model

Save the model and tokeniser via `save_pretrained()`. That writes weights, config, and tokeniser files into one directory that `from_pretrained()` can load again later. The DistilBertSpamClassifier in `ml/predict.py` expects exactly this layout.

In [ ]:
save_dir = 'distilbert_spam_classifier'
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f'Model saved to {save_dir}/')
print(f'\nContents:')
total_size = 0
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(os.path.join(save_dir, f))
    total_size += size
    print(f'  {f} ({size / 1024:.0f} KB)')
print(f'\nTotal size: {total_size / 1e6:.1f} MB')

## 16. Download all outputs

Bundle the trained model and every evaluation file into one zip so a single download covers everything Colab produced. After downloading and unzipping locally, move the files into the project tree:

1. Unzip `distilbert_all_outputs.zip`
2. Move `distilbert_spam_classifier/` to `ml/models/distilbert_spam_classifier/`
3. Move `distilbert_confusion_matrix.png` to `evaluation/`
4. Move `distilbert_roc_curve.png` to `evaluation/`
5. Move `distilbert_training_loss.png` to `evaluation/`
6. Move `distilbert_results.csv` to `evaluation/`

In [ ]:
import zipfile

eval_files = [
    'distilbert_confusion_matrix.png',
    'distilbert_roc_curve.png',
    'distilbert_training_loss.png',
    'distilbert_results.csv'
]

# Create a zip containing everything
zip_name = 'distilbert_all_outputs.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Add model files
    for root, dirs, files_list in os.walk('distilbert_spam_classifier'):
        for f in files_list:
            filepath = os.path.join(root, f)
            zf.write(filepath)
    # Add evaluation files
    for f in eval_files:
        if os.path.exists(f):
            zf.write(f)

zip_size = os.path.getsize(zip_name)
print(f'Created: {zip_name} ({zip_size / 1e6:.1f} MB)')
print(f'\nContents:')
with zipfile.ZipFile(zip_name, 'r') as zf:
    for info in zf.infolist():
        print(f'  {info.filename} ({info.file_size / 1024:.0f} KB)')

print(f'\nAfter downloading, place files as follows:')
print(f'  distilbert_spam_classifier/ → ml/models/distilbert_spam_classifier/')
for f in eval_files:
    print(f'  {f} → evaluation/{f}')

# Trigger browser download
print(f'\nStarting download...')
files.download(zip_name)

## 17. Summary

### Model comparison

| Metric | TF-IDF + LR (baseline) | Fine-tuned DistilBERT |
|--------|----------------------|----------------------|
| Accuracy | 99.09% | *see results above* |
| Precision | 98.53% | *see results above* |
| Recall | 99.71% | *see results above* |
| F1 Score | 99.11% | *see results above* |
| ROC-AUC | 99.91% | *see results above* |
| Training time | < 1 second | ~15 to 25 minutes (GPU) |
| Model size | ~470 KB (pkl files) | ~260 MB |
| Features | 10,000-word TF-IDF | 768-dim contextual embeddings |

### What's actually different about the two models

- Logistic Regression sits on top of a fixed bag-of-words representation (TF-IDF). Each word gets one coefficient, regardless of context. The advantages: fast, tiny, completely interpretable (you can read off feature_importance and know exactly why it made a prediction).
- DistilBERT produces contextual embeddings, where the representation of a word depends on the surrounding words. The same token "free" gets different vectors in "free meeting room" and "free money offer". The advantages: in principle it can pick up patterns the bag-of-words can't, including ordering and negation.

### What's next

Both models score nearly identically on clean data, which is unsurprising and not the point of the dissertation. The actual question, in Phase 6, is what happens when the test set is adversarially modified:
1. Character-level perturbation: typos and internal-character edits in spam keywords.
2. Synonym substitution: replace spam keywords with WordNet synonyms.
3. Good-word injection: pad spam with benign professional vocabulary.

My working hypothesis: DistilBERT's contextual understanding makes it more robust to these attacks than the bag-of-words baseline, which depends on exact word matches. Phase 6 is where I find out whether that hypothesis survives contact with the data.